# Cohort Retention Analysis — Demo

A demonstration of how this portfolio's workflow patterns apply to a marketplace / subscription-program analytics question: **is retention getting better or worse across acquisition cohorts, and why?**

The dataset is synthetic (~10,000 subscribers, twelve monthly cohorts, three countries) — generated by `data/generate_data.js` with a fixed seed so the analysis is reproducible.

This notebook is structured to show the two passes that matter on every analytical question:

1. **What does the headline say?** (the fast, AI-scaffolded first pass)
2. **Does the headline survive five minutes of skepticism?** (the verification habit)

Spoiler: it does not. The headline is a textbook **Simpson's paradox** — aggregate retention is declining while every country's retention is stable. Catching this is the entire point.

## Setup

In [ ]:
import pandas as pd

df = pd.read_csv('data/synthetic-subscriptions.csv')
print(f"Rows: {len(df):,}")
print(f"Users: {df['user_id'].nunique():,}")
print(f"Cohorts: {df['cohort_month'].nunique()}")
print(f"Countries: {df['country'].unique().tolist()}")
print(f"Columns: {df.columns.tolist()}")

Rows: 50,860
Users: 10,000
Cohorts: 12
Countries: ['Country A', 'Country B', 'Country C']
Columns: ['user_id', 'cohort_month', 'country', 'plan_tier', 'channel', 'months_since_acquisition', 'is_retained']

Each row is one user-month observation. We will use **month-3 retention** as the reference metric (standard horizon for early retention quality in subscription businesses — short enough to compare across recent cohorts, long enough to capture early churn).

## Step 1 — First pass with AI scaffolding

I asked Claude to scaffold a cohort retention analysis on this dataset. The prompt and the response below are illustrative of the workflow — in practice this is a five-minute exchange that produces the code in the next cell.

> **Prompt:** "I have a synthetic subscription dataset in long format (one row per user-month, columns: user_id, cohort_month, country, plan_tier, channel, months_since_acquisition, is_retained). Scaffold a cohort retention analysis: for each acquisition cohort, what is the month-3 retention rate? Return both the code and a quick read of the result."

> **Claude's response (paraphrased):** "Filter to month-3 observations, group by cohort_month, average is_retained. The result is the cohort retention curve at the month-3 horizon."

In [ ]:
# AI-scaffolded first pass: aggregate month-3 retention by cohort
m3 = df[df['months_since_acquisition'] == 3]

agg_retention = (
    m3.groupby('cohort_month')['is_retained']
      .agg(['mean', 'count'])
      .rename(columns={'mean': 'retention_rate', 'count': 'n_users'})
)
agg_retention['retention_rate'] = (agg_retention['retention_rate'] * 100).round(1)
print(agg_retention)

             retention_rate  n_users
cohort_month
2025-06                69.4      700
2025-07                73.3      720
2025-08                70.0      750
2025-09                70.6      780
2025-10                69.0      810
2025-11                69.6      830
2025-12                68.6      860
2026-01                67.0      880
2026-02                65.9      900

### Claude's first-pass conclusion

> "Month-3 retention has trended down across cohorts — from **69.4%** in the Jun 2025 cohort to **65.9%** in the Feb 2026 cohort. The drop is roughly 3–4 percentage points over eight months. This points to a retention problem worth investigating: something has changed in the experience of recently-acquired users."

That is a tidy first-pass narrative. It is also where most analyses stop.

## Step 2 — Verification habit: does the headline survive five minutes of skepticism?

Before this conclusion goes into a memo, three questions:

1. Is the aggregate trend consistent **within each segment** (country, channel, plan), or is it driven by one segment?
2. Has the **mix of segments** changed across cohorts? If so, the aggregate trend may be a mix-shift artifact, not a behavior change.
3. Are sample sizes large enough that the differences are not noise?

Question 1 first — break the same metric down by country.

In [ ]:
# Retention at month 3 by cohort and country
by_cohort_country = (
    m3.groupby(['cohort_month', 'country'])['is_retained']
      .mean()
      .unstack()
      .mul(100).round(1)
)
print(by_cohort_country)

country       Country A  Country B  Country C
cohort_month
2025-06            79.7       43.1       54.1
2025-07            78.5       52.8       77.1
2025-08            78.3       53.0       63.0
2025-09            82.4       48.1       67.9
2025-10            78.8       50.8       67.1
2025-11            80.1       49.8       71.4
2025-12            81.0       50.6       67.5
2026-01            78.0       51.8       70.3
2026-02            77.9       50.4       71.1

This changes the picture.

- **Country A:** retention is essentially flat at ~79% across all cohorts.
- **Country B:** retention is essentially flat at ~50% (the 2025-06 outlier is small-sample — n=144).
- **Country C:** noisier, but no consistent trend.

**Within every country, retention is stable.** The aggregate "decline" is not a retention problem.

If the aggregate fell while every component is flat, the only explanation is a **mix shift** between components. That is question 2.

In [ ]:
# Country mix per cohort: % of each cohort coming from each country
mix = m3.groupby(['cohort_month', 'country']).size().unstack()
mix_pct = mix.div(mix.sum(axis=1), axis=0).mul(100).round(1)
print(mix_pct)

country       Country A  Country B  Country C
cohort_month
2025-06            68.9       20.6       10.6
2025-07            70.6       19.7        9.7
2025-08            63.3       26.9        9.7
2025-09            59.6       29.6       10.8
2025-10            60.0       31.4        8.6
2025-11            58.8       31.9        9.3
2025-12            53.8       36.5        9.7
2026-01            50.1       38.4       11.5
2026-02            49.3       41.4        9.2

There it is.

- **Country A** shrinks from **68.9%** of new users to **49.3%** — almost 20 percentage points.
- **Country B** grows from **20.6%** to **41.4%** — doubles its share.
- **Country C** stays roughly stable at ~10%.

Country B retains structurally worse than Country A (50% vs. 79% at month 3). As Country B's share grows, the **mix-weighted average retention drops** even though no individual country's retention is moving.

This is **Simpson's paradox**: a trend that holds in the aggregate reverses or disappears within every subgroup.

## Step 3 — Decomposition: how much of the "decline" is mix shift?

To quantify, hold per-country retention constant at the Jun 2025 levels and recompute the aggregate using each cohort's actual country mix. The difference between this counterfactual and the real aggregate isolates the pure mix-shift effect.

In [ ]:
# Counterfactual: hold each country's retention at the Jun 2025 level,
# but apply each later cohort's country mix. Difference vs. actual aggregate
# = the pure mix-shift contribution.
country_retention_baseline = by_cohort_country.loc['2025-06']  # %
mix_share = mix.div(mix.sum(axis=1), axis=0)                   # fractional

counterfactual = (mix_share * country_retention_baseline).sum(axis=1)

comparison = pd.DataFrame({
    'actual_retention': agg_retention['retention_rate'],
    'counterfactual_mix_only': counterfactual.round(1),
    'gap_vs_2025_06': (agg_retention['retention_rate'] - 69.4).round(1)
})
print(comparison)

             actual_retention  counterfactual_mix_only  gap_vs_2025_06
cohort_month
2025-06                  69.4                     69.4             0.0
2025-07                  73.3                     69.8             3.9
2025-08                  70.0                     67.0             0.6
2025-09                  70.6                     65.8             1.2
2025-10                  69.0                     65.9            -0.4
2025-11                  69.6                     65.3             0.2
2025-12                  68.6                     63.1            -0.8
2026-01                  67.0                     61.8            -2.4
2026-02                  65.9                     60.7            -3.5

The counterfactual (mix-only effect, holding retention constant) tracks the actual aggregate retention closely. The mix shift **explains essentially the entire aggregate decline** — there is no residual retention degradation to investigate at the country level.

If anything, looking at the counterfactual, the actual aggregate retention is **above** what pure mix shift would predict in the later cohorts — meaning within-segment retention may be very slightly improving, masked by the unfavorable mix change.

## Step 4 — Catching this kind of error in AI-scaffolded analysis

A few habits that catch this category of mistake before it ships into an exec memo:

1. **Never report an aggregate trend without breaking it down by the obvious segments.** Country, channel, plan tier, acquisition month. Five minutes of `groupby` saves five days of chasing a phantom problem.
2. **When the aggregate moves but the subgroups do not, suspect a mix shift.** It is the single most common failure mode in cohort-level reporting.
3. **Quantify the decomposition.** Either the mix shift fully explains the trend (as here) or it does not — that distinction changes the recommended action completely.
4. **Verify against an independent definition.** AI will confidently anchor on "retention" as defined in one column. Cross-check by computing retention another way (e.g., from the user_id-level data instead of from the binary flag) to confirm the headline is not an artifact of the column.
5. **Default to "the surprising trend is the data, not the world."** Most surprising aggregate trends in operational data turn out to be measurement or composition effects. The right first move is suspicion, not action.

## Executive Memo

The output of this analysis, in the format that goes to a Head of Pro:

---

**Subject — Cohort retention "decline" is a mix-shift artifact, not a retention problem.**

- Aggregate month-3 retention dropped from **69.4% (Jun-25 cohort)** to **65.9% (Feb-26 cohort)**, ~3.5pp over eight months.
- Within every country, month-3 retention is **stable**: Country A ~79%, Country B ~50%, Country C ~68% across all cohorts. No country shows degradation.
- The aggregate decline is fully explained by a **mix shift in acquisition**: Country A fell from 69% to 49% of new users while Country B (which structurally retains worse) grew from 21% to 41%.
- Recommended action is **not** a retention-experience investigation — it is an acquisition-strategy decision: is the shift toward Country B intentional (growth into a larger market) or unintentional (paid-channel mix change)? The answer determines whether to accept the new aggregate baseline or rebalance acquisition.
- Recommend rebuilding the headline retention metric on a **mix-adjusted basis** going forward, so future cohorts are comparable on retention behavior rather than on retention × acquisition mix.

---

**Five bullets, what changed, what to do.** That is the format that a Head of Pro reads in 90 seconds.